Revenging open ai to extract the relationships between the node entities for each article

In [66]:
import os
import certifi
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get MongoDB credentials from environment variables
MONGO_URI = os.getenv("MONGO_URI")
DB_NAME = os.getenv("MONGO_DB_NAME")
ARTICLES_COLLECTION_NAME = os.getenv("MONGO_ARTICLES_NAME")
NODES_COLLECTION_NAME = os.getenv("MONGO_NODES_NAME")

# Ensure all necessary environment variables are set
if not all([MONGO_URI, DB_NAME, ARTICLES_COLLECTION_NAME, NODES_COLLECTION_NAME]):
    print("❌ Missing required environment variables. Check your .env file.")
    exit()

# Create MongoDB client with TLS verification
try:
    client = MongoClient(MONGO_URI, server_api=ServerApi('1'), tlsCAFile=certifi.where())
    db = client[DB_NAME]

    # Define collections
    articles_collection = db[ARTICLES_COLLECTION_NAME]  # Stores articles
    nodes_collection = db[NODES_COLLECTION_NAME]  # Stores extracted nodes

    # Verify connection
    client.admin.command('ping')
    print(f"✅ Connected to MongoDB Atlas! Database: {DB_NAME}")

except Exception as e:
    print(f"❌ MongoDB Connection Error: {e}")
    exit()

✅ Connected to MongoDB Atlas! Database: tuone


In [67]:
#articles_collection.find_one()

In [68]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()
openAI_api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=openAI_api_key)

def ping_openai(client):
    try:
        response = client.models.list()
        print("✅ Successfully connected to OpenAI API!")
        print(f"Available Models: {[model.id for model in response.data]}")
    except Exception as e:
        print(f"❌ OpenAI API Connection Error: {e}")
ping_openai(client)

✅ Successfully connected to OpenAI API!
Available Models: ['gpt-4o-audio-preview-2024-12-17', 'gpt-4o-realtime-preview-2024-12-17', 'dall-e-3', 'dall-e-2', 'gpt-4o-audio-preview-2024-10-01', 'gpt-4o-realtime-preview-2024-10-01', 'gpt-4o-transcribe', 'gpt-4o-mini-transcribe', 'gpt-4o-realtime-preview', 'babbage-002', 'gpt-4o-mini-tts', 'o1-2024-12-17', 'tts-1-hd-1106', 'text-embedding-3-large', 'gpt-4', 'o1', 'o1-pro-2025-03-19', 'o1-pro', 'computer-use-preview', 'computer-use-preview-2025-03-11', 'tts-1-hd', 'gpt-4o-mini-audio-preview', 'gpt-4o-audio-preview', 'o1-preview-2024-09-12', 'gpt-3.5-turbo-instruct-0914', 'gpt-4o-mini-search-preview', 'tts-1-1106', 'davinci-002', 'gpt-3.5-turbo-1106', 'gpt-4o-search-preview', 'gpt-4-turbo', 'gpt-3.5-turbo-instruct', 'gpt-3.5-turbo', 'gpt-4o-mini-search-preview-2025-03-11', 'gpt-4o-mini-realtime-preview', 'chatgpt-4o-latest', 'whisper-1', 'gpt-3.5-turbo-0125', 'gpt-4-turbo-2024-04-09', 'gpt-3.5-turbo-16k', 'gpt-4o-mini-realtime-preview-2024-12

In [69]:
import json
from bson import json_util
def fetch_articles(limit=00,offset=0):
    try:
        # Fetch articles with an optional limit
        articles_cursor = (articles_collection.find()
                           .skip(offset)
                           .limit(limit))
        articles = list(articles_cursor)

        if articles:
            print(f"✅ Retrieved {len(articles)} articles from MongoDB.\n")
            for idx, article in enumerate(articles, start=1):
                print(f"--- Article {idx} ---")
                # Convert BSON to JSON using json_util
                article_json = json.dumps(article, indent=4, default=json_util.default)
                print(article_json)
        else:
            print("⚠️ No articles found in the collection.")

        return articles

    except Exception as e:
        print(f"❌ Error fetching articles: {e}")
        return []


# Fetch and print articles in JSON format
articles = fetch_articles(limit=5,offset=15)

✅ Retrieved 5 articles from MongoDB.

--- Article 1 ---
{
    "_id": {
        "$oid": "67b1e4240554959fd725ee05"
    },
    "title": "BASF and TODA reinforce battery production in Japan and USA",
    "paragraphs": [
        {
            "p1": "BASF and TODA strengthen their cooperation as they announce to create the world\u2019s largest calcination facility for high nickel Cathode Active Materials in Japan. They will also combine their manufacturing activities in the States.",
            "p2": "BASF TODA Battery Materials LLC (BTBM), a joint venture between BASF and TODA, has now tripled its high nickel Cathode Active Materials (CAMs) capacity at its Onoda site in Japan.",
            "p3": "Jeffrey Lou, Senior Vice President of Battery Materials at BASF said that with the expansion \u201cBASF (is) further strengthening our market position in high energy cathode active materials for e-mobility applications.\u201d",
            "p4": "A similar move is planned in the States were both

In [70]:
def read_prompt_from_file_only(file_path):
    with open(file_path, 'r') as file:
        prompt = file.read()
    return prompt

import json
def load_function_schema(path):
    with open(path, "r") as f:
        return json.load(f)

def combine_paragraphs(article):
    paragraphs = article.get('paragraphs', [])
    # Handle missing or empty paragraphs
    if not paragraphs:
        print("⚠️ No paragraphs found in the article.")
        return ""

    combined_text = ""
    for para_obj in paragraphs:
        for key in sorted(para_obj.keys()):
            combined_text += para_obj[key].strip() + " "

    return combined_text.strip()

In [71]:
def extract_nodes(article_id, text, PROMPT_PATH, FUNCTION_SCHEMA_PATH):

    function_schema = load_function_schema(FUNCTION_SCHEMA_PATH)
    prompt = read_prompt_from_file_only(PROMPT_PATH)

    try:
        completion = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": f"Here is the article: {text}"}
            ],
            functions = [function_schema],
            function_call = {"name": "extract_clean_tech_entities"},
            temperature=0
        )

        # parse the returned function call
        function_args = completion.choices[0].message.function_call.arguments
        extracted_data = json.loads(function_args)
        print(f"✅ Retrieved nodes {extracted_data}\n")

        if "nodes" not in extracted_data:
            raise ValueError("Expected 'nodes' key in LLM output but not found.")

        nodes_by_type = extracted_data["nodes"]

        # Flatten nodes while retaining their type
        formatted_nodes = []
        for node_type, node_list in nodes_by_type.items():
            for node in node_list:
                formatted_node = {
                    "article_id": article_id,
                    "id": node.get("id"),
                    "type": node.get("type"),
                    "name": node.get("name"),
                    "location": {
                        "city": node.get("location", {}).get("city", ""),
                        "country": node.get("location", {}).get("country", "")
                    } if node.get("location") else None,
                    "amount": node.get("amount"),
                    "status": node.get("status"),
                }
                formatted_nodes.append(formatted_node)

        return formatted_nodes

    except json.JSONDecodeError as e:
        print(f"❌ JSON Parsing Error: {e}")
        return []
    except Exception as e:
        print(f"❌ Error extracting nodes: {e}")
        return []

def extract_events(article_id, text, PROMPT_PATH, FUNCTION_SCHEMA_PATH):

    function_schema = load_function_schema(FUNCTION_SCHEMA_PATH)
    prompt = read_prompt_from_file_only(PROMPT_PATH)

    try:
        completion = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": f"Here is the article: {text}"}
            ],
            functions = [function_schema],
            function_call = {"name": "extract_clean_tech_events"},
            temperature=0
        )

        # parse the returned function call
        function_args = completion.choices[0].message.function_call.arguments
        extracted_data = json.loads(function_args)
        print(f"✅ Retrieved nodes {extracted_data}\n")

        if "events" not in extracted_data:
            raise ValueError("Expected 'events' key in LLM output but not found.")

        events = extracted_data["events"]

        # Flatten nodes while retaining their type
        formatted_events = []
        for event in events:
            formatted_events.append({
                "article_id": article_id,
                "id": event.get("id"),
                "event": event.get("event"),
                "date": event.get("date"),
                "type": event.get("type"),
                })

        return formatted_events

    except json.JSONDecodeError as e:
        print(f"❌ JSON Parsing Error: {e}")
        return []
    except Exception as e:
        print(f"❌ Error extracting nodes: {e}")
        return []


In [86]:
def process_articles(limit, offset, PROMPT_PATH, FUNCTION_SCHEMA_PATH, output_path):
    # Retrieve articles with limit & offset, sorted by `_id` for consistency
    articles_to_process = list(
        articles_collection.find(
            {"meta.ID": {"$regex": "^27"}},  # <-- Add this filter
            {"_id": 1, "paragraphs": 1}       # Keep projection as is
        )
        .sort("_id", -1)  # Sort by MongoDB ObjectId (descending)
        .skip(offset)    # Skip first `offset` articles
        .limit(limit)    # Limit the number of articles
    )

    print(f"🔹 Processing {len(articles_to_process)} articles (Offset: {offset})")

    for article in articles_to_process:
        articleID = str(article["_id"])  # Convert ObjectId to string

        # Extract text from paragraphs
        text = combine_paragraphs(article)

        # Skip if no text is extracted
        if not text:
            print(f"⚠️ No valid text found for Article ID: {articleID}. Skipping.")
            continue

        print(f"📌 Processing Article ID: {articleID}")

        try:
            # Extract and format nodes
            formatted_nodes = extract_nodes(articleID, text, PROMPT_PATH, FUNCTION_SCHEMA_PATH)

            # Ensure extracted nodes are valid before insertion
            if not formatted_nodes:
                print(f"⚠️ No nodes extracted for Article ID: {articleID}. Skipping update.")
                continue

            # Update the article document with extracted nodes and combined text
            update_result = articles_collection.update_one(
                {"_id": article["_id"]},  # Match by article ID
                {"$set": {"combined_text": text, output_path: formatted_nodes}}  # Add new fields
            )

            # Log success
            if update_result.modified_count > 0:
                print(f"✅ Updated Article ID: {articleID} with {len(formatted_nodes)} nodes and combined text.")
            else:
                print(f"⚠️ No updates made for Article ID: {articleID}.")

        except Exception as e:
            print(f"❌ Error processing Article ID {articleID}: {e}")

def process_events(limit, offset, PROMPT_PATH, FUNCTION_SCHEMA_PATH, output_path):
    # Retrieve articles with limit & offset, sorted by `_id` for consistency
    articles_to_process = list(
        articles_collection.find(
            {"meta.ID": {"$regex": "^27"}},  # <-- Add this filter
            {"_id": 1, "paragraphs": 1}       # Keep projection as is
        )
        .sort("_id", -1)  # Sort by MongoDB ObjectId (descending)
        .skip(offset)    # Skip first `offset` articles
        .limit(limit)    # Limit the number of articles
    )

    print(f"🔹 Processing {len(articles_to_process)} articles (Offset: {offset})")

    for article in articles_to_process:
        articleID = str(article["_id"])  # Convert ObjectId to string

        # Extract text from paragraphs
        text = combine_paragraphs(article)

        # Skip if no text is extracted
        if not text:
            print(f"⚠️ No valid text found for Article ID: {articleID}. Skipping.")
            continue

        print(f"📌 Processing Article ID: {articleID}")

        try:
            # Extract and format nodes
            formatted_events = extract_events(articleID, text, PROMPT_PATH, FUNCTION_SCHEMA_PATH)

            # Ensure extracted nodes are valid before insertion
            if not formatted_events:
                print(f"⚠️ No events extracted for Article ID: {articleID}. Skipping update.")
                continue

            # Update the article document with extracted nodes and combined text
            update_result = articles_collection.update_one(
                {"_id": article["_id"]},  # Match by article ID
                {"$set": {"combined_text": text, output_path: formatted_events}}  # Add new fields
            )

            # Log success
            if update_result.modified_count > 0:
                print(f"✅ Updated Article ID: {articleID} with {len(formatted_events)} nodes and combined text.")
            else:
                print(f"⚠️ No updates made for Article ID: {articleID}.")

        except Exception as e:
            print(f"❌ Error processing Article ID {articleID}: {e}")

In [99]:
def format_nodes_for_prompt(nodes, allowed_types=None):
    """
    Format nodes into a clear ID-to-name mapping for GPT prompts.
    Optionally filter by a list of allowed node types.
    """
    lines = ["The following is a list of known entities. You MUST ALWAYS use the ID of the event when referring to it in a relationship:"]
    
    for node in nodes:
        node_id = node.get("id")
        node_type = node.get("type")
        name = node.get("name")
        
        if node_id and node_type and name:
            if allowed_types is None or node_type in allowed_types:
                lines.append(f"- {node_id}: {name} ({node_type})")
    
    return "\n".join(lines)

def format_events_for_prompt(events):
    lines = ["The following is a list of known events. You MUST ALWAYS use the ID of the event when referring to it in a relationship:"]
    for event in events:
        event_id = event.get("id")
        event_type = event.get("type")
        event_date = event.get("date")
        if event_id and event_type and event_date:
            lines.append(f"- {event_id}: {event_type} ({event_date})")
    return "\n".join(lines)

def extract_relationships(article_id, text, nodes, PROMPT_PATH, FUNCTION_SCHEMA_PATH, allowed_types=None):
    
    function_schema = load_function_schema(FUNCTION_SCHEMA_PATH)
    prompt = read_prompt_from_file_only(PROMPT_PATH)
    
    compact_nodes = format_nodes_for_prompt(nodes, allowed_types)

    try:
        completion = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": prompt},
                {
                "role": "user",
                 "content": f"""Here is the article text: {text}

                Here is the list of known entities:
                {compact_nodes}

                Please extract only the specified relationship types."""
                }
            ],
            functions=[function_schema],
            function_call={"name": "extract_clean_tech_relationships"},
            temperature=0
        )

        # Parse the returned function call
        function_args = completion.choices[0].message.function_call.arguments
        extracted_data = json.loads(function_args)
        print(f"✅ Retrieved relationships {extracted_data}\n")

        if "relationships" not in extracted_data:
            raise ValueError("Expected 'relationships' key in LLM output but not found.")

        formatted_relationships = []
        for rel in extracted_data["relationships"]:
            formatted_relationships.append({
                "article_id": article_id,
                "source": rel.get("source"),
                "target": rel.get("target"),
                "type": rel.get("type")
            })

        return formatted_relationships

    except json.JSONDecodeError as e:
        print(f"❌ JSON Parsing Error (relationships): {e}")
        return []
    except Exception as e:
        print(f"❌ Error extracting relationships: {e}")
        return []

def extract_relationships_with_events(article_id, text, nodes, events, PROMPT_PATH, FUNCTION_SCHEMA_PATH, allowed_types=None):
    
    function_schema = load_function_schema(FUNCTION_SCHEMA_PATH)
    prompt = read_prompt_from_file_only(PROMPT_PATH)
    
    compact_nodes = format_nodes_for_prompt(nodes, allowed_types)
    print(compact_nodes)
    compact_events = format_events_for_prompt(events)
    print(compact_events)

    try:
        completion = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": prompt},
                {
                "role": "user",
                 "content": f"""Here is the article text: {text}

                Here is the list of known entities:
                {compact_nodes}

                Here is the list of known events:
                {compact_events}

                Please extract only the specified relationship types."""
                }
            ],
            functions=[function_schema],
            function_call={"name": "extract_clean_tech_relationships"},
            temperature=0
        )

        # Parse the returned function call
        function_args = completion.choices[0].message.function_call.arguments
        extracted_data = json.loads(function_args)
        print(f"✅ Retrieved relationships {extracted_data}\n")

        if "relationships" not in extracted_data:
            raise ValueError("Expected 'relationships' key in LLM output but not found.")

        formatted_relationships = []
        for rel in extracted_data["relationships"]:
            formatted_relationships.append({
                "article_id": article_id,
                "source": rel.get("source"),
                "target": rel.get("target"),
                "type": rel.get("type")
            })

        return formatted_relationships

    except json.JSONDecodeError as e:
        print(f"❌ JSON Parsing Error (relationships): {e}")
        return []
    except Exception as e:
        print(f"❌ Error extracting relationships: {e}")
        return []
 
def process_relationships(limit, offset, PROMPT_PATH, FUNCTION_SCHEMA_PATH, relationship_type, allowed_types=None, node_type=None):
    # Query articles that already have nodes_ben and combined_text

    articles_to_process = list(
        articles_collection.find(
            {
                "nodes_ben": {"$exists": True},
                "combined_text": {"$exists": True},
                "meta.ID": {"$regex": "^27"}  # <-- Only this line is added
            },
            {"_id": -1, "combined_text": 1, "nodes_ben": 1, "events_ben": 1}
        )
        .sort("_id", -1)
        .skip(offset)
        .limit(limit)
    )

    print(f"🔹 Processing {len(articles_to_process)} articles for relationship extraction (Offset: {offset})")

    for article in articles_to_process:
        articleID = str(article["_id"])
        text = article.get("combined_text", "")
        nodes = article.get("nodes_ben", [])
        events = article.get("events_ben", [])

        if not text or not nodes:
            print(f"⚠️ Missing text or nodes for Article ID: {articleID}. Skipping.")
            continue

        print(f"📌 Processing Article ID: {articleID}")

        try:
            # Extract relationships using previously extracted nodes
            if node_type == "entities":
                extracted_relationships = extract_relationships(articleID, text, nodes, PROMPT_PATH, FUNCTION_SCHEMA_PATH, allowed_types)
            elif node_type == "events":
                print("PROCESSING EVENTS nodes")
                extracted_relationships = extract_relationships_with_events(articleID, text, nodes, events, PROMPT_PATH, FUNCTION_SCHEMA_PATH, allowed_types)
            else:
                print("node type is not valid")
                return []

            # Ensure a fallback value is set even if empty
            if extracted_relationships is None:
                extracted_relationships = {"relationships": []}

            if not extracted_relationships:
                print(f"⚠️ No relationships extracted for Article ID: {articleID}.")
                continue

            # Update the article document with extracted relationships
            update_result = articles_collection.update_one(
                {"_id": article["_id"]},
                {"$set": {relationship_type: extracted_relationships}}
            )

            if update_result.modified_count > 0:
                print(f"✅ Updated Article ID: {articleID} with {len(extracted_relationships)} relationships.")
            else:
                print(f"⚠️ No update performed for Article ID: {articleID}.")

        except Exception as e:
            print(f"❌ Error processing relationships for Article ID {articleID}: {e}")


In [ ]:
n_articles = 4

# named entity recognition (like company, factory, investment)
PROMPT_PATH = "prompts/entities-only.txt"
FUNCTION_SCHEMA_PATH = "schemas/entities.json"

process_articles(n_articles,0, PROMPT_PATH, FUNCTION_SCHEMA_PATH, 'nodes_ben')

# extract events that happen (new-announcemment, construction-start)
PROMPT_PATH = "prompts/events.txt"
FUNCTION_SCHEMA_PATH = "schemas/events.json"

#process_events(n_articles,0, PROMPT_PATH, FUNCTION_SCHEMA_PATH, 'events_ben')

# extract ownership relationships
PROMPT_PATH = "prompts/relations-ownership.txt"
FUNCTION_SCHEMA_PATH = "schemas/relations-ownership.json"

#process_relationships(n_articles,0, PROMPT_PATH, FUNCTION_SCHEMA_PATH, "relationships_ownership", allowed_types = ["Company", "Joint Venture", "Factory"])

# extract financial relationships (origin)
PROMPT_PATH = "prompts/relations-financial-origin.txt"
FUNCTION_SCHEMA_PATH = "schemas/relations-financial-origin.json"

#process_relationships(n_articles,0, PROMPT_PATH, FUNCTION_SCHEMA_PATH, "relationships_financial_origin", allowed_types = ["Company", "Joint Venture", "Investment"])

# extract financial relationships (technological)
PROMPT_PATH = "prompts/relations-financial-technological.txt"
FUNCTION_SCHEMA_PATH = "schemas/relations-financial-technological.json"

#process_relationships(n_articles,0, PROMPT_PATH, FUNCTION_SCHEMA_PATH, "relationships_financial_technological", allowed_types = ["Investment", "Capacity", "Product", "Factory"])

# extract technological relationships
PROMPT_PATH = "prompts/relations-technological.txt"
FUNCTION_SCHEMA_PATH = "schemas/relations-technological.json"

#process_relationships(n_articles,0, PROMPT_PATH, FUNCTION_SCHEMA_PATH, "relationships_technological", allowed_types = ["Capacity", "Product", "Factory"])

# extract chronology of factory events
PROMPT_PATH = "prompts/chronology-factory.txt"
FUNCTION_SCHEMA_PATH = "schemas/chronology-factory.json"

#process_relationships(n_articles,0, PROMPT_PATH, FUNCTION_SCHEMA_PATH, "relationships_chronology_factory", allowed_types = ["Factory"], node_type = "events")

# extract chronology of investment events
PROMPT_PATH = "prompts/chronology-investment.txt"
FUNCTION_SCHEMA_PATH = "schemas/chronology-investment.json"

#process_relationships(n_articles,0, PROMPT_PATH, FUNCTION_SCHEMA_PATH, "relationships_chronology_investment", allowed_types = ["Investment"], node_type = "events")

# extract chronology of capacity events
PROMPT_PATH = "prompts/chronology-capacity.txt"
FUNCTION_SCHEMA_PATH = "schemas/chronology-capacity.json"

process_relationships(n_articles,0, PROMPT_PATH, FUNCTION_SCHEMA_PATH, "relationships_chronology_capacity", allowed_types = ["Capacity"], node_type = "events")


🔹 Processing 4 articles (Offset: 0)
📌 Processing Article ID: 67b20c230554959fd7264b95
✅ Retrieved nodes {'nodes': {'companies': [{'id': 'company_1', 'type': 'Company', 'name': 'Volkswagen'}, {'id': 'company_2', 'type': 'Company', 'name': 'IG Metall'}], 'joint_ventures': [], 'factories': [{'id': 'factory_1', 'type': 'Factory', 'name': 'Zwickau electric car plant', 'location': {'city': 'Zwickau', 'country': 'Germany'}}, {'id': 'factory_2', 'type': 'Factory', 'name': 'VW plant in Puebla', 'location': {'city': 'Puebla', 'country': 'Mexico'}}, {'id': 'factory_3', 'type': 'Factory', 'name': 'main plant in Wolfsburg', 'location': {'city': 'Wolfsburg', 'country': 'Germany'}}, {'id': 'factory_4', 'type': 'Factory', 'name': 'Emden plant', 'location': {'city': 'Emden', 'country': 'Germany'}}], 'investments': [], 'capacities': [], 'products': [{'id': 'product_1', 'type': 'Product', 'name': 'ID.3'}, {'id': 'product_2', 'type': 'Product', 'name': 'ID.4'}, {'id': 'product_3', 'type': 'Product', 'name

In [75]:
articles_collection.find_one({'title': 'Tesla to start production of the Model Y Juniper in January'})

{'_id': ObjectId('67b20c220554959fd7264b94'),
 'title': 'Tesla to start production of the Model Y Juniper in January',
 'paragraphs': [{'p1': "Tesla has been producing the Model Y for around five years. So it's time for a facelift, which some market observers had expected as early as 2024. Now the time has supposedly come - Tesla is planning to start mass production of the revised Model Y, codenamed Juniper, at its plant in Shanghai in January.",
   'p2': 'However, there has been no official confirmation from Tesla so far. In addition, reports from Chinese media only refer to the factory in China. However, the Model Y is also built in Grünheide near Berlin, and in the US factories in California and Texas. There is still no indication of a corresponding production changeover at these locations.',
   'p3': 'According to reports from China, Tesla will revise the Model Y in a similar way to theModel 3 Highland facelift. That includes improvements to the interior, exterior, battery capacity